## Configuration

In [ ]:
from pathlib import Path
import tomllib
from utils.utils import *
import json

PATH_EEG = Path("../EEG/EEG/results/")
FILENAME = 'Case_11_ExG.csv'
MARKER_FILE = 'Case_11_Marker.csv'
CONFIG_PATH = "../EEG/EEG/config/config.toml"

with open(CONFIG_PATH, "rb") as f:
    config = tomllib.load(f)

BANDS = config['BANDS']
SEGMENT_DEF = config['SEGMENT_DEF']
RANDOM_SEED = config['RANDOM_SEED']
DURATION = config['DURATION']

SFREQ = config['SFREQ']
L_FREQ = config['L_FREQ']
H_FREQ = config['H_FREQ']

## Raw Data

In [ ]:
raw_full, df_full, df_markers_full = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)

raws, labels = extract_segments(raw_full, df_full, df_markers_full, SEGMENT_DEF)

In [ ]:
plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha')
plot_topomap_comparison(raws, labels, band_name='Beta')

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta')
plot_topomap_comparison(raws, labels, band_name='Theta')

In [ ]:
# from matplotlib import pyplot as plt
# %matplotlib qt  

ANNOTATION_FILE = PATH_EEG / f"{FILENAME}_bad_segments.csv"

# fig = raw.plot(duration=15, n_channels=16, block=True)

# raw.annotations.save(ANNOTATION_FILE, overwrite=True)

In [ ]:
import os
import mne
raw, df, df_markers = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)
if os.path.exists(ANNOTATION_FILE):
    print(f"Load annotations from: {ANNOTATION_FILE}")
    saved_annotations = mne.read_annotations(ANNOTATION_FILE)
    
    aligned_annotations = mne.Annotations(
        onset=saved_annotations.onset,
        duration=saved_annotations.duration,
        description=saved_annotations.description,
        orig_time=raw.annotations.orig_time
    )
    
    raw.set_annotations(raw.annotations + aligned_annotations)
    print("SUCCESS: Annotations merged successfully!")
    
else:
    print("There are no annotations file. Clear it by hand")
    
    %matplotlib qt  

    # Pamiętaj, żeby zaznaczać 'BAD_movement' lub podobne
    fig = raw.plot(duration=15, n_channels=16, block=True)
    
    raw.annotations.save(ANNOTATION_FILE)
    print(f"Annotations saved to: {ANNOTATION_FILE}")
    
%matplotlib inline

## Preprocessing

In [ ]:
import mne
from mne.preprocessing import ICA

# ICA
# n_components=15 (16 Channels, ICA finding max n_channels - 1 or same number)
ica = ICA(n_components=15, max_iter='auto', random_state=RANDOM_SEED)

print(">>> ICA (Might take a while)...")
ica.fit(raw)

print(">>> Showing Components...")
ica.plot_components()
plt.show()

ica.plot_sources(raw, show_scrollbars=True)
plt.show()

In [ ]:
# ica.plot_properties(raw, picks=range(15))

In [ ]:
exclude_indices = [0, 2, 4, 6, 14] 

print(f">>> Removing ingredients: {exclude_indices}")
ica.exclude = exclude_indices

raw_clean = raw.copy()
ica.apply(raw_clean)

print(">>> Cleaned data.")

target_chan = 'F3' if 'F3' in raw.ch_names else raw.ch_names[0]

print(f"Comparison on the channel {target_chan}...")
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(raw.get_data(picks=target_chan)[0, 1000:2500], color='red', alpha=0.5, label='Before cleaning (Original)')
ax.plot(raw_clean.get_data(picks=target_chan)[0, 1000:2500], color='black', label='After cleaning (Clean)')
ax.set_title(f"Artifact Removal Effect (Eyes and Muscles) - Channel {target_chan}")
ax.legend()
plt.show()

In [ ]:
raws, labels = extract_segments(raw_clean, df_full, df_markers_full, SEGMENT_DEF)

plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha')
plot_topomap_comparison(raws, labels, band_name='Beta')

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta')
plot_topomap_comparison(raws, labels, band_name='Gamma')

#### Analiza Pasma Alpha (8–12 Hz):
Na mapach nie widać dysocjacji w pasmie centralnym (C3/C4). Cała kora ruchowa i czołowa pozostaje całkowicie wyciszona (granatowa) we wszystkich czterech trybach. Uczestnik jest w stanie ciągłej, równej gotowości motorycznej.

**Faza 1 (Brainrot) i Faza 2 (Smart)**: Bardzo silna, skondensowana Alpha z tyłu głowy (ciemnoczerwona plama nad O1, Oz, O2). Kiedy uczestnik nie przewija ekranu, jego kora wzrokowa wchodzi w stan "uśpienia" / biernego chłonięcia obrazu.

**Fazy 3 i 4 (Ze Scrollem)**: Fizyczny akt przewinięcia minimalnie wybudza wzrok (czerwień w potylicy robi się odrobinę bledsza). Mózg musi na ułamek sekundy wyjść z transu, by zdekodować nową klatkę.

#### Analiza Pasma Beta (13–30 Hz):
Cały przód i środek mózgu pozostaje całkowicie nieaktywny (granatowy). Potężna, skondensowana aktywność (Czerwień) występuje wyłącznie w korze potylicznej we wszystkich trybach.

Uczestnik odcina się od bodźców słuchowych czy analizy logicznej w płatach czołowych. Najsilniejsza aktywacja Bety pojawia się w Fazie 1 (Brainrot) – próba wyłuskania detali z wideo bez możliwości przewijania zmusza jego korę wzrokową do najwyższego wysiłku analitycznego.

#### Analiza Pasma Delta (1–4 Hz):
**Faza 1 (Brainrot) i Faza 4 (Smart Scroll)**: Mózg jest "przytomny".

**Faza 2 (Smart) i Faza 3 (Brainrot Scroll)**: Pojawia się nagła, intensywna czerwona plama (wysoka Delta) nad lewą korą potyliczną (O1).

To punktowe wyczerpanie analizą wizualną. Bierne czytanie gęstego tekstu (Smart) oraz ciągłe miganie pędzących po scrollu obrazków (Brainrot Scroll) drastycznie męczy lewą półkulę wzrokową. Kiedy jednak może ten "mądry" tekst samemu przewijać we własnym tempie (Smart Scroll), wyczerpanie natychmiast znika.

#### Analiza Pasma Gamma (>30 Hz):
**Brainrot (Faza 1)**: "pożar" w potylicy.

To koszt fizjologiczny. Pasywne oglądanie szybko zmieniających się, jaskrawych wideo zmusza jego mózg do pracy integracyjnej, by skleić klatki w logiczną całość.

W każdym innym trybie (szczególnie w fazach ze scrollem) Gamma drastycznie spada.

## Spektogram

In [ ]:
# Alpha (fatigue/relax)
plot_band_power_over_time(raws, labels, band=(8, 12), band_name='Alpha')
# Beta (Focus)
plot_band_power_over_time(raws, labels, band=(13, 30), band_name='Beta')

In [ ]:
# Emotion
calculate_asymmetry(raws, labels, ch_left='F3', ch_right='F4')

# According to Heller's model, parietal asymmetry is responsible for physiological arousal
calculate_asymmetry(raws, labels, ch_left='P3', ch_right='P4')

# Image processing method.
calculate_asymmetry(raws, labels, ch_left='O1', ch_right='O2')

# Motor
calculate_asymmetry(raws, labels, ch_left='C3', ch_right='C4')

### Podsumowanie

#### Kora Czołowa (F4 vs F3)
Mózg ani przez sekundę nie czuje przyjemności. Wskaźnik FAA poniżej zera to dowód na negatywny afekt, znużenie i chęć wycofania się z sytuacji.

To, co na topomapach widać było jako "pożar" w potylicie (wysoka Gamma i Beta), w połączeniu z ujemnymi wartościami na czole, daje obraz skrajnego przebodźcowania. Badany nie czerpie radości z Brainrotu – on jest nim bombardowany i jego mózg desperacko chce przestać to oglądać.

#### Kora Ruchowa (C4 vs C3)
Skoro C4-C3 < 0, oznacza to, że prawa półkula ruchowa jest znacznie bardziej aktywna.

Pacjent jest leworęczny lub obsługiwał telefon lewą ręką. Lewa ręka pracuje na pełnych obrotach, co idealnie widać w każdym trybie scrollowania.

#### Kora Ciemieniowa (P4 vs P3)
Dominacja prawej kory ciemieniowej.

Prawa strona ciemieniowa odpowiada za przetwarzanie bodźców budzących lęk lub niepokój przestrzenny. Ujemny wynik sugeruje, że uczestnik czuje się osaczony. Próbuje odnaleźć się w gąszczu szybko zmieniających się obrazów.

#### Kora Potyliczna (O2 vs O1)
Dominacja prawej półkuli wzrokowej (O2).

Badany przetwarza obraz holistycznie (całościowo), ale w kontekście minusów na czole nie jest to "spokojne oglądanie". To raczej wizualne oszołomienie. Mózg chłonie obraz jako jedną, chaotyczną masę, nie potrafiąc przejść do lewopółkulowej analizy detali czy czytania napisów.

## Scroll

In [ ]:
PROJECT_ROOT = Path.cwd().parent 

USER_MAP_PATH = PROJECT_ROOT / "EEG" / "EEG" / "user_map.json"
INTERACTIONS_PATH = PROJECT_ROOT / "EEG" / "research_events_rows.csv"
USERS_PATH = PROJECT_ROOT / "EEG" / "Users_rows.csv"

FILE_NAME = 'Case_11_ExG.csv'

In [ ]:
scroll_data = extract_scroll_events(FILE_NAME, INTERACTIONS_PATH, USERS_PATH, USER_MAP_PATH)

if scroll_data is not None and not scroll_data.empty:
    new_annotations = mne.Annotations(
        onset=scroll_data['onset'].values,
        duration=scroll_data['duration'].values,
        description=scroll_data['description'].values
    )

    raw_clean.set_annotations(new_annotations)
    
    print("SUCCESS: Annotations applied to raw_clean.")
    print(f"Time range of scrolls: {scroll_data['onset'].min():.2f}s to {scroll_data['onset'].max():.2f}s")
else:
    print("WARNING: No scroll data found. Plots will be empty.")

In [ ]:
compare_stages(
    raw_clean, 
    band='Beta', 
    fmin=13, 
    fmax=30, 
    channel='C3',  
    duration=DURATION
)

In [ ]:
# REWARD/MOTIVATION SYSTEM (Frontal Alpha)
# Channel: F3 (Left Frontal)
# Band: Alpha (8-12 Hz)
# Interpretation: THE LOWER THE GRAPH, THE GREATER THE “REWARD/ENGAGEMENT.”

compare_stages(
    raw_clean, 
    band='Alpha', 
    fmin=8, 
    fmax=12, 
    channel='F3',   # Key channel for positive/aspirational emotions
    duration=DURATION
)

In [ ]:
# ISUAL RELAXATION (Alpha)
# Channel: O1 (Back of the head)
# Band: Alpha (8-12 Hz)
# Interpretation: HIGH LINE = RELAXATION / ZOMBIE MODE. LOW = WORK.

compare_stages(
    raw_clean, 
    band='Alpha', 
    fmin=8, 
    fmax=12, 
    channel='O1', 
    duration=DURATION
)

In [ ]:
# INTELLECTUAL EFFORT (Frontal Theta) ---
# Channel: Fz (Center of forehead)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = STRONG THINKING/MEMORIZATION.

target_channel = 'Fz'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# COGNITIVE CONTROL & INHIBITION (Right Frontal Theta) ---
# Channel: F4 (Right forehead / Right Frontal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = SUSTAINED FOCUS & SUPPRESSING DISTRACTIONS (Mental effort to stay on task or regulate emotional impulses).

target_channel = 'F4'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# VERBAL & ANALYTICAL MEMORY (Left Parietal Theta) ---
# Channel: P3 (Left back of head / Left Parietal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = PROCESSING TEXT/LANGUAGE (Reading captions, understanding speech, verbal working memory).

target_channel = 'P3'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# VISUOSPATIAL PROCESSING & ATTENTION (Right Parietal Theta) ---
# Channel: P4 (Right back of head / Right Parietal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = ENCODING VISUAL/SPATIAL INFO (Processing layout, movement, spatial memory, or complex imagery).

target_channel = 'P4'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

#### Kora wzrokowa (Alpha O1)
**Brainrot Scroll**: Wykres wykazuje wysokie piki pomiędzy poszczególnymi czerwonymi liniami. Gdy tylko następuje scroll (czerwona linia), moc Alfy gwałtownie załamuje się do zera, by zaraz po nim wystrzelić w górę w formie chaotycznego "szoku". Mózg nie nadąża za tempem obrazu.

**Smart Scroll**: Choć średnia moc jest podobna, spadki po niebieskich liniach są głębsze i trwają dłużej. Sugeruje to, że przy treściach Smart badany próbuje "uchwycić" obraz, ale kosztuje go to ogromny wysiłek koncentracyjny.

#### Kora ruchowa (Beta C3)
**Smart Scroll**: Zamiast cyklu przygotowania do ruchu, widać tu serię nerwowych, drobnych drgań trendu Beta. Nawet przy rzadkim scrollowaniu, kora ruchowa nie wchodzi w stan relaksu po wykonaniu gestu.

**Brainrot Scroll**: Linia trendu jest niemal płaska i "poszarpana" na niskim poziomie. Badany przewija ekran odruchowo, bez świadomego planowania ruchu, co skutkuje brakiem naturalnych oscylacji fali Beta towarzyszących celowym działaniom.

#### Płaty czołowe (Theta Fz, F4)
**Smart Scroll**: Po każdej niebieskiej linii następuje niemal natychmiastowy, bardzo ostry pik Theta na Fz. To nie jest "spokojne zrozumienie", to raczej skokowy wysiłek poznawczy. Mózg musi gwałtownie "wskoczyć" na wysokie obroty, żeby przetworzyć treść, co jest dla niego męczące.

**Brainrot Scroll**: Wykresy Fz i F4 są rozregulowane. Piki Theta pojawiają się losowo, często niezależnie od samych czerwonych linii scrollowania. To stan, w którym płat czołowy "poddaje się" i nie jest w stanie synchronizować uwagi z tym, co palec robi na ekranie.

#### Płaty ciemieniowe (Theta P3, P4)
**Brainrot Scroll**: Moc Theta na P4 (prawa półkula – stres) jest wyższa niż na P3 (lewa półkula).

**Wniosek**: Każdy szybki scroll w trybie Brainrot generuje silniejszą odpowiedź w prawej korze ciemieniowej, co jest fizjologicznym sygnałem dezorientacji i lęku przestrzennego. Badany czuje się zagubiony w tym, co widzi na ekranie.

---
Pacjent to przykład osoby, dla której interfejs smartfona jest wrogiem.

Każde przesunięcie palcem to wizualny szok (O1) i skokowy stres poznawczy (Fz).

Mechaniczna praca kory ruchowej (C3) bez naturalnych cykli odbicia sugeruje, że czynność ta nie sprawia mu żadnej przyjemności.

Przewaga mocy Theta na P4 nad P3 podczas szybkich scrolli potwierdza, że badany fizycznie źle znosi dynamikę wideo.